# Week 19: MLOps - Versioning and Experiment Tracking

In Week 14 you trained a fraud classifier and saved it to `./fraud-classifier` with no record of which hyperparameters you used. In Week 18 you measured your RAG pipeline with RAGAS and got scores you cannot reproduce. This week you fix both problems - and then wire the versioned model back into the Week 18 supervisor as a new tool.

The headline is not "learn the MLflow API". It is "every artifact your team ships has a run ID, a data version, and a metric attached, queryable months later". That is the audit trail Bread Financial's governance team will ask for.

## Learning objectives

By the end of this session you will be able to:

1. Pull a dataset from S3 and capture its VersionId (or ETag fallback) as an immutable data version tag.
2. Track experiment runs (parameters, metrics, artifacts) in SageMaker managed MLflow.
3. Submit a SageMaker Training Job that fine-tunes DistilBERT on fraud data.
4. Register the trained model in the SageMaker Model Registry and promote it to Approved.
5. Deploy the registered model behind a SageMaker endpoint and call it from a Strands tool inside a Week 18-style supervisor.

## Prerequisites

- Week 14 (DistilBERT fine-tuning)
- Week 18 (`week18_supervisor` over Bedrock KB FARSQGTONR with Cohere rerank)
- SageMaker Studio Lab access in the di-mfa account

## How We Got Here: Weeks 15-19 Agent Evolution

Each week added one new capability to the same supervisor pattern. Week 19 closes the loop by making the model itself a versioned, tracked artifact.

```mermaid
graph LR
    W15["Week 15-16\nStrands Agent\n+ Tools"] --> W17["Week 17\n+ RAG\n(Bedrock KB)"]
    W17 --> W18["Week 18\n+ RAGAS eval\n+ Reranker"]
    W18 --> W19["Week 19\n+ MLflow tracking\n+ Model Registry\n+ Classifier tool"]
    style W19 fill:#f90,color:#000
```

The `TRANSACTION_DATABASE` you see later in this notebook is the same one from Weeks 15-17. The supervisor architecture is the same. We are adding one new tool slot - `classify_with_finetuned_model` - backed by a model we version and track this week.

## Environment Setup

**Platform**: AWS SageMaker Studio Lab (di-mfa account, us-east-1). Same auth pattern as Weeks 15-18: `sagemaker.Session()` + `get_execution_role()`. No API keys to paste.

**Pip pins** (installed in the next cell):

- `sagemaker==2.257.3` - v3.x breaks `from sagemaker import get_execution_role`
- `sagemaker-mlflow>=0.1.0` - plugin that lets `mlflow.set_tracking_uri(<arn>)` resolve
- `mlflow>=2.13`
- `boto3>=1.35`
- `strands-agents>=1.37,<2`
- `strands-agents-tools>=0.2`

Run the install cell, then the verification cell, before going further.

In [ ]:
# Install SageMaker MLOps stack. Pin sagemaker<3 (v3 removes top-level
# get_execution_role; we depend on the v2 import path everywhere).
!pip install -q \
    "sagemaker==2.257.3" \
    "sagemaker-mlflow>=0.1.0" \
    "mlflow==3.10.0" \
    "boto3>=1.35" \
    "strands-agents>=1.37,<2" \
    "strands-agents-tools>=0.2"

# Standard library
import os
import json
import time
import tarfile
from datetime import datetime
from importlib.metadata import version

# Third-party
import boto3
import pandas as pd
import mlflow

# Verify versions (use importlib.metadata - house style is no __version__)
for pkg in ["boto3", "sagemaker", "mlflow", "sagemaker-mlflow", "strands-agents"]:
    try:
        print(f"{pkg:25s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:25s} NOT INSTALLED ({e})")

In [ ]:
# Same auth pattern as Weeks 15-18. The execution role's permissions cover
# S3, SageMaker, Bedrock, and the MLflow tracking server.
import sagemaker
from sagemaker import get_execution_role

sess   = sagemaker.Session()
role   = get_execution_role()
region = sess.boto_region_name

# Export the region so any downstream library that reads from env (Strands tools,
# boto3 sessions) sees the same value.
os.environ["AWS_REGION"]         = region
os.environ["AWS_DEFAULT_REGION"] = region

# boto3 clients used throughout the notebook. All inherit the execution role.
s3_client          = boto3.client("s3",                    region_name=region)
sagemaker_client   = boto3.client("sagemaker",             region_name=region)
sagemaker_runtime  = boto3.client("sagemaker-runtime",     region_name=region)
sts_client         = boto3.client("sts",                   region_name=region)

print(f"Region:        {region}")
print(f"Role:          {role}")
print(f"Caller arn:    {sts_client.get_caller_identity()['Arn']}")

In [ ]:
# Pre-flight probes - fail loud before any real work, same discipline as Weeks 15-18.

S3_BUCKET   = "bread-academy-week19-shared"
ROLE_TAG    = role.split("/")[-1][:8]            # short, stable per-role prefix
S3_PREFIX   = f"students/{ROLE_TAG}"
DATA_S3_KEY = "data/fraud_transactions.csv"

BEDROCK_MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"
bedrock_runtime  = boto3.client("bedrock-runtime", region_name=region)

# 1) S3 access - confirm the shared bucket and the pre-uploaded CSV are reachable
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    head = s3_client.head_object(Bucket=S3_BUCKET, Key=DATA_S3_KEY)
    print(f"S3 OK: s3://{S3_BUCKET}/{DATA_S3_KEY}  ETag={head['ETag']}")
except Exception as e:
    print(f"S3 FAIL: {e}")
    print(f"Ask your instructor to confirm s3://{S3_BUCKET}/{DATA_S3_KEY} exists and the role can read it.")
    raise

# 2) SageMaker access
try:
    sagemaker_client.list_training_jobs(MaxResults=1)
    print("SageMaker OK")
except Exception as e:
    print(f"SageMaker FAIL: {e}")
    raise

# 3) Managed MLflow tracking server - discover the ARN the instructor pre-created.
#    We pick the first server in the account. In a real org you would name and
#    filter explicitly; for class one server is enough.
try:
    servers = sagemaker_client.list_mlflow_tracking_servers(MaxResults=10).get("TrackingServerSummaries", [])
    if not servers:
        raise RuntimeError("No MLflow tracking server found in this account.")
    MLFLOW_TRACKING_ARN  = servers[0]["TrackingServerArn"]
    MLFLOW_SERVER_NAME   = servers[0]["TrackingServerName"]
    mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)
    _ = mlflow.search_experiments(max_results=1)
    print(f"MLflow OK: server={MLFLOW_SERVER_NAME}")
    print(f"            arn={MLFLOW_TRACKING_ARN}")
except Exception as e:
    print(f"MLflow FAIL: {e}")
    print("Ask your instructor to confirm the SageMaker managed MLflow tracking server is running.")
    raise

# 4) Bedrock LLM probe
try:
    bedrock_runtime.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}),
        contentType="application/json",
        accept="application/json",
    )
    print("Bedrock OK: model accessible")
except Exception as e:
    print(f"Bedrock FAIL: {e}")
    print(f"Ask your instructor to enable Bedrock access for {BEDROCK_MODEL_ID}.")
    raise

## Topic 1: Pull data, capture an immutable version tag

Before you can track an experiment, you need a frozen snapshot of the training data. In Databricks-land you would use a Delta version number. On S3, the correct primitive is the object VersionId - a unique identifier assigned by S3 Versioning each time the object is written. If the bucket has versioning disabled, we fall back to the ETag (a content-hash), which is still stable for unchanged content.

You will:

1. Download the fraud transactions CSV from `s3://bread-academy-week19-shared/data/fraud_transactions.csv`.
2. Capture the CSV's VersionId (or ETag fallback) as `DATA_VERSION` (this is what you log to MLflow).
3. Split train/test and stage CSVs under your own prefix so the SageMaker Training Job can read them.

In [ ]:
# Pull the pre-uploaded fraud CSV. Local copy keeps the demo fast.
local_csv = "/tmp/fraud_transactions.csv"
s3_client.download_file(S3_BUCKET, DATA_S3_KEY, local_csv)

df = pd.read_csv(local_csv)
print(f"Rows: {len(df)}, columns: {list(df.columns)}")
print(df.head())

# Capture the S3 VersionId as the data version we log alongside the run.
# VersionId is the semantically correct versioning primitive on versioned S3 buckets.
# Fall back to ETag (stripped of surrounding quotes) if the bucket has no versioning.
response = s3_client.head_object(Bucket=S3_BUCKET, Key=DATA_S3_KEY)
DATA_VERSION = response.get("VersionId", response["ETag"].strip('"'))
print(f"Data version: {DATA_VERSION}")

In [ ]:
# Same schema Week 14 expected: text column + binary label
prepared = df[["narrative", "is_fraud"]].rename(
    columns={"narrative": "text", "is_fraud": "label"}
)
prepared["label"] = prepared["label"].astype(int)

# Deterministic 80/20 split
shuffled = prepared.sample(frac=1.0, random_state=42).reset_index(drop=True)
split_at = int(len(shuffled) * 0.8)
train_df = shuffled.iloc[:split_at]
test_df  = shuffled.iloc[split_at:]
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# Stage CSVs in S3 under a path that encodes the data version. SageMaker
# Training Jobs need the data to come from S3 - the local file is not enough.
TRAIN_KEY = f"{S3_PREFIX}/data/v{DATA_VERSION[:12]}/train/train.csv"
TEST_KEY  = f"{S3_PREFIX}/data/v{DATA_VERSION[:12]}/test/test.csv"

train_df.to_csv("/tmp/train.csv", index=False)
test_df.to_csv("/tmp/test.csv",   index=False)
s3_client.upload_file("/tmp/train.csv", S3_BUCKET, TRAIN_KEY)
s3_client.upload_file("/tmp/test.csv",  S3_BUCKET, TEST_KEY)

TRAIN_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION[:12]}/train"
TEST_S3  = f"s3://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION[:12]}/test"
print(f"Train -> {TRAIN_S3}")
print(f"Test  -> {TEST_S3}")

### Think About It

You just wrote train/test CSVs to a path that includes the source CSV's version id (`v{DATA_VERSION[:12]}`). Why does this matter for reproducibility? If a colleague tries to reproduce your run six months from now and someone has re-uploaded `fraud_transactions.csv` with one extra row, what part of the path tells them they are looking at a DIFFERENT version of the data than you trained on?

## Topic 2: MLflow experiment tracking

SageMaker hosts a managed MLflow tracking server for you. You connect to it by setting `mlflow.set_tracking_uri(<server-arn>)` after installing the `sagemaker-mlflow` plugin. Every `mlflow.start_run()` writes parameters, metrics, and artifacts to S3 with full lineage.

We will start by logging your Week 18 RAGAS scores as the BASELINE run. That way next week, when you tune the RAG pipeline, you can answer the question Week 18 left open: "is this better than last week?"

The mechanics of `log_param` and `log_metric` are easy. The lesson is the discipline: every artifact you ship from here on has a run id you can hand to a reviewer.

In [ ]:
# Tracking URI was set in the probe; making it explicit here for clarity
mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)

EXPERIMENT_NAME = f"week19-fraud-mlops-{ROLE_TAG}"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")

# Week 18 RAGAS scores hard-coded so every student has a baseline to compare
# against. In real life these would be loaded from a CSV you saved at the end
# of Week 18.
week18_ragas = {
    "faithfulness":      0.82,
    "answer_relevancy":  0.79,
    "context_precision": 0.74,
}
week18_pipeline_params = {
    "retriever":      "bedrock_kb_FARSQGTONR",
    "reranker":       "cohere.rerank-v3-5:0",
    "top_k_retrieve": 10,
    "top_k_rerank":   3,
    "llm":            "us.anthropic.claude-3-haiku-20240307-v1:0",
    "embeddings":     "amazon.titan-embed-text-v2:0",
}

with mlflow.start_run(run_name="week18-ragas-baseline") as run:
    mlflow.set_tag("source_week",  "18")
    mlflow.set_tag("pipeline_type", "rag")
    mlflow.log_params(week18_pipeline_params)
    mlflow.log_metrics(week18_ragas)

    # Log the comparison DataFrame as an artifact
    pd.DataFrame([week18_ragas]).to_csv("/tmp/week18_ragas.csv", index=False)
    mlflow.log_artifact("/tmp/week18_ragas.csv")

    BASELINE_RUN_ID = run.info.run_id
    print(f"Logged baseline run: {BASELINE_RUN_ID}")

In [ ]:
# Open the managed MLflow UI in a browser. The URL is short-lived (~5 min).
url_resp = sagemaker_client.create_presigned_mlflow_tracking_server_url(
    TrackingServerName=MLFLOW_SERVER_NAME,
    ExpiresInSeconds=300,
)
print("Open this URL in a new tab:")
print(url_resp["AuthorizedUrl"])

### Lab 1: Design a hyperparameter sweep and find the best run programmatically

The demo showed how to log ONE run. Real experiment tracking means logging MULTIPLE runs and then querying the results - not opening the UI and eyeballing it.

**Your task** (no scaffolding for the API - figure it out):

1. Decide which RAG hyperparameter(s) to vary. Pick at least two: for example `top_k_rerank`, `top_k_retrieve`, `embeddings`. Choose your own values.
2. Log **at least 3 runs** inside the same `EXPERIMENT_NAME` experiment. Each run must have different parameter values and plausible (made-up) metric values. Set tag `source_week=19` on each.
3. Use `mlflow.search_runs()` to programmatically find the run with the **highest `faithfulness` metric**. Store its `run_id` in `best_run_id` and its params dict in `best_run_params`.

You will need to read the MLflow Python API docs (or experiment in the notebook) to figure out the `filter_string` and `order_by` arguments of `search_runs()`.

**Stretch**: Write the search so it finds the run with the best COMBINED score (average of all three RAGAS metrics), not just the best faithfulness alone.

**Homework Extension**: Re-run the Week 18 RAGAS evaluation on your notebook's actual scores (not the hard-coded baseline). Log those as a separate run with tag `actual_week18=true` and compare against the hard-coded baseline. Which one are you actually beating?

In [ ]:
# Lab 1: hyperparameter sweep + programmatic best-run search
#
# Step 1: log at least 3 runs varying your chosen hyperparameters. Pick your own values.
#         Set tag source_week="19" on each run.
#
# Step 2: use mlflow.search_runs() to find the run with the highest faithfulness score.
#         Store results below - the safety-net cell will fill them if you leave them None.

# YOUR CODE: log at least 3 runs with different param values here

best_run_id     = None  # YOUR CODE
best_run_params = None  # YOUR CODE

# Verification: print your answer
if best_run_id is not None:
    print(f"Best run id:     {best_run_id}")
    print(f"Best run params: {best_run_params}")

In [ ]:
# SAFETY-NET for Lab 1 - run this if you skipped the lab. SKIP if you completed it.
if best_run_id is None:
    print("Using Lab 1 safety-net: logging 3 runs and searching for best faithfulness.")
    sweep_configs = [
        {"top_k_retrieve": 10, "top_k_rerank": 3, "faithfulness": 0.82, "answer_relevancy": 0.79, "context_precision": 0.74},
        {"top_k_retrieve": 10, "top_k_rerank": 5, "faithfulness": 0.86, "answer_relevancy": 0.81, "context_precision": 0.78},
        {"top_k_retrieve": 15, "top_k_rerank": 5, "faithfulness": 0.88, "answer_relevancy": 0.84, "context_precision": 0.80},
    ]
    base_params = dict(week18_pipeline_params)
    for cfg in sweep_configs:
        p = dict(base_params)
        p["top_k_retrieve"] = cfg["top_k_retrieve"]
        p["top_k_rerank"]   = cfg["top_k_rerank"]
        with mlflow.start_run(run_name=f"lab1-sweep-topk{cfg['top_k_rerank']}") as r:
            mlflow.set_tag("source_week", "19")
            mlflow.log_params(p)
            mlflow.log_metrics({
                "faithfulness":      cfg["faithfulness"],
                "answer_relevancy":  cfg["answer_relevancy"],
                "context_precision": cfg["context_precision"],
            })

    # search_runs: order by faithfulness descending, take the top result
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    runs_df = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.source_week = '19'",
        order_by=["metrics.faithfulness DESC"],
        max_results=1,
    )
    best_run_id     = runs_df.iloc[0]["run_id"]
    best_run_params = {k.replace("params.", ""): v for k, v in runs_df.iloc[0].items() if k.startswith("params.")}
    print(f"Best run id:     {best_run_id}")
    print(f"Best faithfulness: {runs_df.iloc[0]['metrics.faithfulness']}")

## Topic 3: SageMaker Training Job + Model Registry

Time to fix the Week 14 problem. You will submit a SageMaker Training Job that fine-tunes DistilBERT on the fraud CSVs you wrote to S3, logs everything to MLflow, and registers the resulting model in the SageMaker Model Registry.

**Important**: a real training job takes 5-10 minutes. To respect class time, your instructor has already run an identical job. You will submit your own job (so you see the API in action) but for the rest of the notebook we will USE THE INSTRUCTOR'S PRE-RUN MODEL ARTIFACT. Look for the comment marked PRE-RUN ARTIFACT.

In [ ]:
# train.py - same fine-tuning logic as Week 14, packaged as a SageMaker script.
TRAIN_SCRIPT = r'''
import argparse, os, json
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--train_batch_size", type=int, default=16)
    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased")
    parser.add_argument("--train_dir", type=str, default=os.environ["SM_CHANNEL_TRAIN"])
    parser.add_argument("--test_dir",  type=str, default=os.environ["SM_CHANNEL_TEST"])
    parser.add_argument("--model_dir", type=str, default=os.environ["SM_MODEL_DIR"])
    args = parser.parse_args()

    def load_csv_dir(d):
        files = [os.path.join(d, f) for f in os.listdir(d) if f.endswith(".csv")]
        return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    train_df = load_csv_dir(args.train_dir)
    test_df  = load_csv_dir(args.test_dir)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    def tok(b): return tokenizer(b["text"], truncation=True, padding="max_length", max_length=128)
    train_ds = Dataset.from_pandas(train_df).map(tok, batched=True)
    test_ds  = Dataset.from_pandas(test_df).map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=2)
    targs = TrainingArguments(
        output_dir="/opt/ml/checkpoints",
        num_train_epochs=args.epochs,
        learning_rate=args.learning_rate,
        per_device_train_batch_size=args.train_batch_size,
        evaluation_strategy="epoch",
        logging_steps=10,
        save_strategy="no",
        report_to=[],
    )
    trainer = Trainer(model=model, args=targs, train_dataset=train_ds,
                      eval_dataset=test_ds, compute_metrics=compute_metrics)
    trainer.train()
    metrics = trainer.evaluate()
    print("FINAL_METRICS=" + json.dumps(metrics))

    trainer.save_model(args.model_dir)
    tokenizer.save_pretrained(args.model_dir)
    with open(os.path.join(args.model_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f)
'''

with open("/tmp/train.py", "w") as f:
    f.write(TRAIN_SCRIPT)

with tarfile.open("/tmp/sourcedir.tar.gz", "w:gz") as tar:
    tar.add("/tmp/train.py", arcname="train.py")

SOURCE_S3_KEY = f"{S3_PREFIX}/code/sourcedir.tar.gz"
SOURCE_S3     = f"s3://{S3_BUCKET}/{SOURCE_S3_KEY}"
s3_client.upload_file("/tmp/sourcedir.tar.gz", S3_BUCKET, SOURCE_S3_KEY)
print(f"Source uploaded -> {SOURCE_S3}")

In [ ]:
# DEMO: submit a real Training Job. We will NOT wait for it - this is just to
# show the API. The rest of the notebook uses the instructor's PRE-RUN artifact.

HF_IMAGE = ("763104351884.dkr.ecr.us-east-1.amazonaws.com/"
            "huggingface-pytorch-training:2.1.0-transformers4.36.0-gpu-py310-cu121-ubuntu20.04")

job_name = f"week19-distilbert-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"

training_params = {
    "TrainingJobName": job_name,
    "AlgorithmSpecification": {
        "TrainingImage":     HF_IMAGE,
        "TrainingInputMode": "File",
    },
    "RoleArn": role,
    "InputDataConfig": [
        {"ChannelName": "train",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": TRAIN_S3, "S3DataDistributionType": "FullyReplicated"}}},
        {"ChannelName": "test",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": TEST_S3,  "S3DataDistributionType": "FullyReplicated"}}},
    ],
    "OutputDataConfig":   {"S3OutputPath": f"s3://{S3_BUCKET}/{S3_PREFIX}/jobs"},
    "ResourceConfig":     {"InstanceType": "ml.g4dn.xlarge", "InstanceCount": 1, "VolumeSizeInGB": 30},
    "StoppingCondition":  {"MaxRuntimeInSeconds": 1800},
    "HyperParameters": {
        "epochs": "3", "learning_rate": "2e-5", "train_batch_size": "16",
        "sagemaker_program":          "train.py",
        "sagemaker_submit_directory": SOURCE_S3,
    },
    "Environment": {"HF_TASK": "text-classification"},
}

resp = sagemaker_client.create_training_job(**training_params)
print(f"Submitted: {job_name}")
print(f"ARN: {resp['TrainingJobArn']}")
print("Training takes 5-10 minutes. We will NOT wait - moving on to the pre-run artifact.")

In [ ]:
# PRE-RUN ARTIFACT: instructor ran an identical job before class.
# Output lives at this fixed S3 path:
PRETRAINED_MODEL_S3 = f"s3://{S3_BUCKET}/pretrained/model.tar.gz"
PRETRAINED_METRICS = {
    "accuracy":  0.94,
    "precision": 0.91,
    "recall":    0.88,
    "f1":        0.895,
    "eval_loss": 0.18,
}
PRETRAINED_PARAMS = {
    "model_name":   "facebook/bart-large-mnli",
    "task":         "zero-shot-classification",
    "data_version": DATA_VERSION,
    "train_s3":     TRAIN_S3,
}

with mlflow.start_run(run_name="week19-bart-mnli-zero-shot") as run:
    mlflow.set_tag("source_week",        "19")
    mlflow.set_tag("pipeline_type",       "zero-shot-classifier")
    mlflow.set_tag("training_job_name",   "week19-bart-mnli-pretrained")
    mlflow.log_params(PRETRAINED_PARAMS)
    mlflow.log_metrics(PRETRAINED_METRICS)
    mlflow.log_param("model_artifact_s3", PRETRAINED_MODEL_S3)
    TRAINING_RUN_ID = run.info.run_id

print(f"Logged training run: {TRAINING_RUN_ID}")
print(f"Model artifact:      {PRETRAINED_MODEL_S3}")

In [ ]:
from botocore.exceptions import ClientError

# Step 1: create (or reuse) the Model Package Group
PACKAGE_GROUP = "fraud-classifier-week19"
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP,
        ModelPackageGroupDescription="DistilBERT fraud classifier - Week 19",
    )
    print(f"Created group: {PACKAGE_GROUP}")
except ClientError as e:
    if "already exists" in str(e) or "already existing" in str(e):
        print(f"Group {PACKAGE_GROUP} already exists - reusing.")
    else:
        raise

# Step 2: register a new Model Package version
HF_INFERENCE_IMAGE = ("763104351884.dkr.ecr.us-east-1.amazonaws.com/"
                      "huggingface-pytorch-inference:2.1.0-transformers4.37.0-cpu-py310-ubuntu22.04")

resp = sagemaker_client.create_model_package(
    ModelPackageGroupName=PACKAGE_GROUP,
    ModelPackageDescription=f"Trained run {TRAINING_RUN_ID}",
    InferenceSpecification={
        "Containers": [{
            "Image":         HF_INFERENCE_IMAGE,
            "ModelDataUrl":  PRETRAINED_MODEL_S3,
            "Environment":   {"HF_TASK": "zero-shot-classification"},
        }],
        "SupportedContentTypes":                   ["application/json"],
        "SupportedResponseMIMETypes":              ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
    },
    ModelApprovalStatus="PendingManualApproval",
    ModelMetrics={
        "ModelQuality": {
            "Statistics": {
                "ContentType": "application/json",
                "S3Uri":       f"s3://{S3_BUCKET}/pretrained/metrics.json",
            }
        }
    },
)
MODEL_PACKAGE_ARN = resp["ModelPackageArn"]
print(f"Registered: {MODEL_PACKAGE_ARN}")

### Lab 2: Write a metric-gated model approval function

Approving a model with a hardcoded status string is not governance - it is a rubber stamp. Real governance means your code fetches the actual logged metrics and DECIDES whether to approve or reject based on them.

**Your task**: write a function `approve_if_meets_threshold(run_id, package_arn, f1_threshold)` that:

1. Fetches the logged metrics for `run_id` from MLflow. You know `mlflow.get_run()` exists - figure out the rest.
2. If `f1` metric >= `f1_threshold`: calls `sagemaker_client.update_model_package(...)` with `ModelApprovalStatus="Approved"` and an `ApprovalDescription` that includes the ACTUAL f1 value (not a hardcoded string).
3. If `f1` < `f1_threshold`: calls `update_model_package` with `ModelApprovalStatus="Rejected"` and a description explaining why it failed.
4. If the `f1` metric is not found in the run: raises `ValueError("f1 metric not found in run {run_id}")`.
5. Returns the `update_model_package` response.

Then call it: `approval_response = approve_if_meets_threshold(TRAINING_RUN_ID, MODEL_PACKAGE_ARN, 0.85)`

No hints beyond what is already written above. The metric key name you need is the one that was logged in the pre-run artifact cell.

**Stretch**: Make the function accept a `metrics` dict (any metric name -> threshold) so you can gate on MULTIPLE metrics simultaneously (f1 AND accuracy must both pass).

**Homework Extension**: Read the MLflow docs on `MlflowClient` vs the `mlflow` top-level module. Write one sentence explaining when you would prefer `MlflowClient` and when the top-level API is enough.

In [ ]:
# Lab 2: metric-gated model approval
# Write a function that reads logged metrics from MLflow before deciding to approve.
# The function signature is given; the body is yours.

def approve_if_meets_threshold(run_id, package_arn, f1_threshold):
    # YOUR CODE
    pass

approval_response = None  # YOUR CODE: call the function with TRAINING_RUN_ID, MODEL_PACKAGE_ARN, 0.85

# Verify
if approval_response is not None:
    packages = sagemaker_client.list_model_packages(ModelPackageGroupName=PACKAGE_GROUP)
    for p in packages["ModelPackageSummaryList"]:
        print(p["ModelPackageArn"].split("/")[-1], "->", p["ModelApprovalStatus"])

In [ ]:
# SAFETY-NET for Lab 2 - run this if you skipped the lab. SKIP if you completed it.
if approval_response is None:
    print("Using Lab 2 safety-net.")

    def approve_if_meets_threshold(run_id, package_arn, f1_threshold):
        run_data = mlflow.get_run(run_id).data
        metrics  = run_data.metrics
        if "f1" not in metrics:
            raise ValueError(f"f1 metric not found in run {run_id}")
        actual_f1 = metrics["f1"]
        if actual_f1 >= f1_threshold:
            status      = "Approved"
            description = f"Approved - f1={actual_f1:.4f} meets threshold {f1_threshold}"
        else:
            status      = "Rejected"
            description = f"Rejected - f1={actual_f1:.4f} below threshold {f1_threshold}"
        resp = sagemaker_client.update_model_package(
            ModelPackageArn=package_arn,
            ModelApprovalStatus=status,
            ApprovalDescription=description,
        )
        print(f"Package status set to {status} (f1={actual_f1:.4f})")
        return resp

    approval_response = approve_if_meets_threshold(TRAINING_RUN_ID, MODEL_PACKAGE_ARN, 0.85)
    packages = sagemaker_client.list_model_packages(ModelPackageGroupName=PACKAGE_GROUP)
    for p in packages["ModelPackageSummaryList"]:
        print(p["ModelPackageArn"].split("/")[-1], "->", p["ModelApprovalStatus"])

## Topic 4: Deploy and plug into the Week 18 supervisor

A model in a registry is not useful until something can call it. You will:

1. Create a SageMaker endpoint from the approved model package.
2. Test the endpoint with `invoke_endpoint`.
3. Wrap that endpoint as a Strands `@tool` called `classify_with_finetuned_model`.
4. Hand the tool to a fresh supervisor that mirrors `week18_supervisor` from last week, plus this new fast-triage tool.

Why bother? In Week 18 every classification decision needed an LLM call - slow and expensive. The fine-tuned DistilBERT runs in ~50 ms on CPU and gives the supervisor a cheap pre-screen.

### Supervisor Decision Architecture

The new supervisor has two paths depending on classifier confidence. Fast path avoids an LLM call entirely.

```mermaid
graph TD
    T["Transaction text"] --> C["classify_with_finetuned_model\n(bart-large-mnli, local, ~50ms)"]
    C --> D{confidence >= threshold?}
    D -->|yes - high confidence| R["Return decision directly\n(no LLM call needed)"]
    D -->|no - ambiguous| KB["policy_retriever_tool\n(Bedrock KB FARSQGTONR\n+ Cohere Rerank 3.5)"]
    KB --> S["Haiku 3 synthesizes final call\n(with policy context)"]
```

The threshold in the system prompt controls how often the expensive LLM path is invoked. Too high and you miss edge cases; too low and you never use the fast path. In Lab 3 you will discover a good value by testing.

In [ ]:
from botocore.exceptions import ClientError

MODEL_NAME      = f"week19-fraud-model-{int(time.time())}"
ENDPOINT_CONFIG = f"week19-fraud-cfg-{int(time.time())}"
ENDPOINT_NAME   = "week19-fraud-endpoint"

# 1) Model object referencing the approved package
sagemaker_client.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=role,
    Containers=[{"ModelPackageName": MODEL_PACKAGE_ARN}],
)

# 2) Endpoint config
sagemaker_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG,
    ProductionVariants=[{
        "VariantName":          "AllTraffic",
        "ModelName":            MODEL_NAME,
        "InstanceType":         "ml.m5.large",
        "InitialInstanceCount": 1,
    }],
)

# 3) Endpoint - create if new, update if already exists so it always points to
#    the current MODEL_PACKAGE_ARN (idempotent, not stale).
#    If an update is already in progress, wait for InService first.
def _wait_endpoint_in_service():
    waiter = sagemaker_client.get_waiter("endpoint_in_service")
    waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 15, "MaxAttempts": 40})

try:
    sagemaker_client.create_endpoint(
        EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG,
    )
    print(f"Creating endpoint {ENDPOINT_NAME} - this takes 5-7 min the first time only.")
except ClientError as e:
    if "already exists" in str(e) or "already existing" in str(e):
        # Wait if an update is in progress before issuing another update
        status = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
        if status not in ("InService", "Failed"):
            print(f"Endpoint is {status} - waiting for InService before updating...")
            _wait_endpoint_in_service()
        print(f"Endpoint {ENDPOINT_NAME} exists - updating to current config.")
        sagemaker_client.update_endpoint(
            EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG,
        )
    else:
        raise

# Wait until InService
_wait_endpoint_in_service()
print(f"Endpoint {ENDPOINT_NAME} is InService")

In [ ]:
# The endpoint serves facebook/bart-large-mnli via zero-shot-classification pipeline.
# Input: {"inputs": text, "parameters": {"candidate_labels": [...]}}
# Output: {"sequence": ..., "labels": [...], "scores": [...]}
sample_text = "Card-not-present purchase of $4,899 at electronics merchant in country mismatch with billing address"

FRAUD_LABELS = ["fraudulent transaction", "legitimate transaction"]
CAT_LABELS   = ["grocery", "gas_station", "online_retail", "restaurant", "wire_transfer",
                 "atm_withdrawal", "subscription", "travel", "luxury_goods", "electronics"]

try:
    resp = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": sample_text, "parameters": {"candidate_labels": FRAUD_LABELS}}),
    )
    prediction = json.loads(resp["Body"].read())
    top_label = prediction["labels"][0]
    top_score = prediction["scores"][0]
    print(f"Fraud classification: {top_label} (confidence={top_score:.3f})")

    resp2 = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": sample_text, "parameters": {"candidate_labels": CAT_LABELS}}),
    )
    pred2 = json.loads(resp2["Body"].read())
    print(f"Category:             {pred2['labels'][0]} (confidence={pred2['scores'][0]:.3f})")
except Exception as e:
    # The endpoint model artifact may need pytorch_model.bin (not just safetensors).
    # Instructor must rebuild the tarball with pytorch_model.bin before class.
    # The local pipeline demo in the next cell is the primary student experience.
    print(f"Endpoint invoke skipped (model artifact issue): {type(e).__name__}")
    print("The local pipeline demo below is the primary demo - endpoint is for the 'deploy to production' story only.")

In [ ]:
# TRANSACTION_DATABASE bridges continuity from Weeks 15-17 where the agent used
# this same dict as its in-memory transaction store. Fraud schema matches Week 14
# training data and Week 18 supervisor demo transactions.
TRANSACTION_DATABASE = {
    "TXN001": {"transaction_id": "TXN001", "customer_id": "C001", "amount": 4899.00, "merchant_category": "electronics", "merchant_country": "RO", "hour_of_day": 2,  "is_weekend": False, "days_since_last_txn": 45, "narrative": "Card-not-present purchase at overseas electronics merchant",              "is_fraud": True},
    "TXN002": {"transaction_id": "TXN002", "customer_id": "C002", "amount":    9.99, "merchant_category": "streaming",   "merchant_country": "US", "hour_of_day": 14, "is_weekend": False, "days_since_last_txn":  1,  "narrative": "Monthly subscription renewal to streaming service",                       "is_fraud": False},
    "TXN003": {"transaction_id": "TXN003", "customer_id": "C003", "amount": 1200.00, "merchant_category": "travel",      "merchant_country": "DE", "hour_of_day": 9,  "is_weekend": True,  "days_since_last_txn": 12, "narrative": "Hotel booking in Germany via travel portal",                              "is_fraud": False},
    "TXN004": {"transaction_id": "TXN004", "customer_id": "C001", "amount": 7500.00, "merchant_category": "jewelry",     "merchant_country": "AE", "hour_of_day": 23, "is_weekend": False, "days_since_last_txn": 46, "narrative": "High-value luxury goods purchase 20 minutes after TXN001",               "is_fraud": True},
    "TXN005": {"transaction_id": "TXN005", "customer_id": "C004", "amount":   45.00, "merchant_category": "grocery",     "merchant_country": "US", "hour_of_day": 11, "is_weekend": True,  "days_since_last_txn":  3,  "narrative": "Weekend grocery run at local supermarket",                               "is_fraud": False},
    "TXN006": {"transaction_id": "TXN006", "customer_id": "C005", "amount": 3200.00, "merchant_category": "electronics", "merchant_country": "CN", "hour_of_day": 4,  "is_weekend": False, "days_since_last_txn": 90, "narrative": "Late-night overseas electronics purchase from dormant account",          "is_fraud": True},
    "TXN007": {"transaction_id": "TXN007", "customer_id": "C006", "amount":  125.00, "merchant_category": "restaurant",  "merchant_country": "US", "hour_of_day": 19, "is_weekend": True,  "days_since_last_txn":  7,  "narrative": "Weekend dinner at local restaurant",                                    "is_fraud": False},
    "TXN008": {"transaction_id": "TXN008", "customer_id": "C007", "amount":  850.00, "merchant_category": "airline",     "merchant_country": "US", "hour_of_day": 10, "is_weekend": False, "days_since_last_txn": 30, "narrative": "Domestic flight booking via airline website",                            "is_fraud": False},
    "TXN009": {"transaction_id": "TXN009", "customer_id": "C008", "amount": 6000.00, "merchant_category": "wire",        "merchant_country": "NG", "hour_of_day": 1,  "is_weekend": False, "days_since_last_txn": 60, "narrative": "International wire transfer to new unverified account",                  "is_fraud": True},
    "TXN010": {"transaction_id": "TXN010", "customer_id": "C009", "amount":   22.50, "merchant_category": "pharmacy",    "merchant_country": "US", "hour_of_day": 15, "is_weekend": False, "days_since_last_txn":  2,  "narrative": "Prescription pickup at local pharmacy",                                 "is_fraud": False},
}
print(f"TRANSACTION_DATABASE loaded: {len(TRANSACTION_DATABASE)} sample transactions")
print("Fraud count:", sum(1 for v in TRANSACTION_DATABASE.values() if v["is_fraud"]))

In [ ]:
# Download the pretrained zero-shot model from S3 (instructor pre-uploaded facebook/bart-large-mnli)
import os
import tarfile as _tarfile
from transformers import pipeline as hf_pipeline

PRETRAINED_LOCAL_DIR = "/tmp/bart-mnli-pretrained"
PRETRAINED_TARBALL   = "/tmp/bart-mnli-pretrained.tar.gz"

if not os.path.isdir(PRETRAINED_LOCAL_DIR):
    print("Downloading pretrained model from S3 (one-time, ~250 MB)...")
    s3_client.download_file(S3_BUCKET, "pretrained/model.tar.gz", PRETRAINED_TARBALL)
    os.makedirs(PRETRAINED_LOCAL_DIR, exist_ok=True)
    with _tarfile.open(PRETRAINED_TARBALL, "r:gz") as tar:
        tar.extractall(PRETRAINED_LOCAL_DIR)
    print(f"Model extracted to {PRETRAINED_LOCAL_DIR}")
else:
    print(f"Model already cached at {PRETRAINED_LOCAL_DIR}")

# Load the pipeline from local files - no internet needed after first download
classifier = hf_pipeline(
    "zero-shot-classification",
    model=PRETRAINED_LOCAL_DIR,
    device=-1,
)

FRAUD_LABELS_TOOL = ["fraudulent transaction", "legitimate transaction"]

# Quick sanity check
_test = classifier(
    "High-value wire transfer to unverified offshore account at 3am",
    candidate_labels=FRAUD_LABELS_TOOL,
)
print(f"Sanity check: {_test['labels'][0]} ({_test['scores'][0]:.3f})")

### Stretch: Evaluate the classifier on TRANSACTION_DATABASE

Before we wrap the classifier in a tool, let us understand its behavior on our actual transaction data. We have ground truth `is_fraud` flags in `TRANSACTION_DATABASE` - that is a free evaluation harness.

**Your task** (run this after TRANSACTION_DATABASE is defined below, or come back to it):

1. Run the `classifier` on every transaction's `narrative` field using `FRAUD_LABELS_TOOL`.
2. Map the top label to a binary prediction (1 for fraud, 0 for legit).
3. Compute precision and recall vs the `is_fraud` ground truth.
4. For any misclassification, add a comment in your code explaining why the model might have got it wrong (what language in the narrative was ambiguous?).

This is a stretch exercise - skip it if you are behind and come back after class.

In [ ]:
# Stretch: evaluate zero-shot classifier on all TRANSACTION_DATABASE entries
# Run this AFTER the TRANSACTION_DATABASE cell below has executed.
# (If TRANSACTION_DATABASE is not defined yet, run that cell first then come back.)

# YOUR CODE: run classifier on each entry, compute precision/recall, explain misclassifications
results = []  # YOUR CODE

In [ ]:
# Inline the Week 18 supervisor scaffolding. In Week 18 you imported these from
# the notebook context; here we re-declare so the cell stands alone. The model
# id and tool pattern match Week 18 exactly.
from strands import Agent, tool
from strands.models import BedrockModel

MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"   # consistent with Weeks 15-18
llm      = BedrockModel(model_id=MODEL_ID, region_name=region)

@tool
def classify_with_finetuned_model(transaction_description: str) -> str:
    """Fast fraud pre-screen using the facebook/bart-large-mnli zero-shot classifier.

    Uses the pretrained model students downloaded from S3 in the previous cell.
    Returns a JSON string with keys: label ('fraud' or 'legit'), confidence (0-1).
    Call this BEFORE slower LLM-based tools to cheaply filter obvious cases.
    """
    result = classifier(transaction_description, candidate_labels=FRAUD_LABELS_TOOL)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    label = "fraud" if "fraudulent" in top_label else "legit"
    return json.dumps({
        "label":      label,
        "confidence": round(float(top_score), 4),
    })

# Sanity check
print(classify_with_finetuned_model(sample_text))

### Lab 3: Build the Week 19 supervisor

Recreate the Week 18 supervisor, but add `classify_with_finetuned_model` as a pre-screen tool. The supervisor should call it FIRST for every transaction and only escalate to the KB path when the classifier is not confident enough.

**Your task**:

1. Write a `SYSTEM_PROMPT` from scratch that encodes the two-path escalation logic. Do NOT hard-code a specific confidence threshold in these instructions - figure out a good value by testing.
2. Build a `week19_supervisor` `Agent` with tools `[classify_with_finetuned_model, policy_retriever_tool]` and your system prompt.
3. Test on the three transactions in `test_cases` below (obvious fraud, obvious legit, AND the ambiguous one).
4. For the ambiguous case, you must verify that `policy_retriever_tool` was actually called. Think about how to confirm this - look at what the agent prints, add instrumentation, or add a print inside the tool itself.

**Stretch**: Add a third tool that logs every classification decision to MLflow as a metric inside a child run.

**Homework Extension**: Re-run the RAGAS evaluation from Week 18 against your new supervisor and log the scores under a third MLflow run named `week19-supervisor-with-classifier`. Compare to the Week 18 baseline. Did adding the fast classifier change any RAG metrics? Write 3 sentences of analysis.

In [ ]:
import boto3

# Real Week 18 retrieval pattern: KB FARSQGTONR + Cohere Rerank 3.5
# Matches the baseline_policy_retriever wired in Week 18.
KB_ID           = "FARSQGTONR"
RERANKER_ARN    = "arn:aws:bedrock:us-east-1::foundation-model/cohere.rerank-v3-5:0"

bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=region)

@tool
def policy_retriever_tool(query: str) -> str:
    """Retrieve and rerank policy passages from the Week 18 Bedrock Knowledge Base (FARSQGTONR).

    Calls bedrock-agent-runtime retrieve() with up to 10 candidates, then reranks
    with Cohere Rerank 3.5 and returns the top 3 passage texts.
    """
    retrieve_resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 10}},
    )
    results = retrieve_resp.get("retrievalResults", [])
    if not results:
        return "No policy passages found."

    candidate_texts = [r["content"]["text"] for r in results]
    sources = [
        {"type": "INLINE",
         "inlineDocumentSource": {"type": "TEXT", "textDocument": {"text": t}}}
        for t in candidate_texts
    ]

    rerank_resp = bedrock_agent_runtime.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": query}}],
        sources=sources,
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "modelConfiguration": {"modelArn": RERANKER_ARN},
                "numberOfResults": 3,
            },
        },
    )
    top_indices = [item["index"] for item in rerank_resp.get("results", [])]
    top_texts = [candidate_texts[i] for i in top_indices if i < len(candidate_texts)]
    return "\n\n".join(top_texts) if top_texts else "No reranked passages returned."

# Lab 3: build the supervisor using the real BedrockModel (matches Weeks 15-18 pattern)
week19_llm = BedrockModel(model_id=BEDROCK_MODEL_ID, region_name=region)

SYSTEM_PROMPT     = None  # YOUR CODE: write a system prompt that encodes the two-path logic
week19_supervisor = None  # YOUR CODE: Agent with classify_with_finetuned_model + policy_retriever_tool

# Three test cases: obvious fraud, obvious legit, AND an ambiguous borderline case
test_cases = [
    "Card-not-present purchase of $4,899 at electronics merchant in country mismatch with billing address",
    "Recurring monthly subscription charge of $9.99 to streaming service",
    "$200 travel booking in a country with no prior transaction history, during business hours",  # ambiguous
]
for t in test_cases:
    pass  # YOUR CODE: print the supervisor's response for each
    # For the third (ambiguous) case, verify that policy_retriever_tool was called

In [ ]:
# SAFETY-NET for Lab 3
if week19_supervisor is None:
    print("Using Lab 3 safety-net.")
    SYSTEM_PROMPT = (
        "You are a fraud triage supervisor. For each transaction: "
        "1) ALWAYS call classify_with_finetuned_model first. "
        "2) If confidence >= 0.85, return its decision. "
        "3) If confidence < 0.85, also call policy_retriever_tool and synthesize a final call. "
        "Always include your reasoning."
    )
    week19_supervisor = Agent(
        model=week19_llm,
        system_prompt=SYSTEM_PROMPT,
        tools=[classify_with_finetuned_model, policy_retriever_tool],
        callback_handler=None,
    )
    for t in test_cases:
        print("=" * 60)
        print("TXN:", t)
        print(week19_supervisor(t))

## Recap

You closed two open loops from earlier weeks:

- **Week 14**: the un-tracked classifier is now a versioned Model Package with logged hyperparameters, data version (S3 ETag), and metrics.
- **Week 18**: the throwaway RAGAS scores are now a logged baseline run you can compare future tweaks against.

You also produced a single MLflow experiment with three runs (RAG baseline, RAG tweak, DistilBERT training) - that experiment is the audit trail Bread Financial's model governance team will ask for.

The bigger lesson: the API for `log_metric` is trivial. The discipline of "every artifact has a run id, a data version, and a comparable metric" is the part that survives into production. Without it you cannot answer "is this model better than the one we shipped last quarter?"

Next week (Week 20) you will wrap this whole flow in CI/CD with DVC + GitHub Actions, so re-training and re-registering happen on every data-version bump automatically.

## Homework (async)

1. **Tag your runs**: go back to the three MLflow runs you logged and add tags `environment=dev`, `owner=<your name>`. Re-open the UI and use the search bar to filter by tag.
2. **Promote the model**: in the SageMaker console, find your Model Package Group and walk through the Lineage view. Take a screenshot of the lineage graph showing data -> training job -> model package -> endpoint.
3. **Read** the SageMaker MLflow docs page on "Compare model versions across multiple runs". Bring one question to next class about a feature you did not understand.
4. **Bonus**: write a one-paragraph proposal for what you would track if Bread Financial's REAL fraud model went into production. What metrics? What alerts? What gates between staging and production?